# Corrected Qwen judge — Colab Pro

**Reusable template. The live L4 readiness probe passed on October 7, 2026; every new runtime must pass its own checks.** This notebook fine-tunes the judge; it does not train a foundation model from scratch or change the answerlab product.

1. Choose **Runtime → Change runtime type → GPU**. Use a BF16-capable GPU (for example L4 or A100); the code checks this and stops if unsupported. Availability depends on your account/session.
2. Upload `Judge_Colab_Package.zip` below. It contains public tokenized HelpSteer2 training/validation data, no model weights or credentials.
3. Install the text-only dependencies, then use **Runtime → Restart session**. Resume at the tests/Drive cell; do not repeat the upload cell.
4. Run the short probe. Only after it passes, inspect speed/memory and enable the bounded sweep.
5. Evaluate unchanged Qwen and the selected adapter using the same float32 A/B head.

Calibration and final-test data are absent. No long training starts by default. Public data: NVIDIA HelpSteer2, CC BY 4.0, revision `990b2711a36180dd19d9c94b8627844866f8982a`. Treat candidate text as data, never instructions.

In [ ]:
from pathlib import Path
import zipfile, json, hashlib
root = Path("/content/judge_colab").resolve()
if not root.exists():
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one Judge_Colab_Package ZIP."
    archive = Path(next(iter(uploaded)))
    assert zipfile.is_zipfile(archive), "The uploaded file must be a ZIP."
    with zipfile.ZipFile(archive) as package:
        assert "package_manifest.json" in package.namelist(), "Missing package manifest."
        for name in package.namelist():
            assert (root / name).resolve().is_relative_to(root), "Unsafe archive path."
        package.extractall(root)
else:
    print("Reusing existing extracted package; checking its fingerprints below.")
manifest = json.loads((root / "package_manifest.json").read_text())
for name, expected in manifest.items():
    path = (root / name).resolve()
    assert path.is_relative_to(root), "Unsafe manifest path."
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, "Changed package file: " + name
print("Verified", len(manifest), "package files in", root)


In [ ]:
from pathlib import Path
import subprocess, sys
root = Path("/content/judge_colab")
# These optional image/audio/quantization add-ons are unused by this text-only, unquantized judge.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "torchao"], check=True, timeout=120)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(root / "code/requirements.txt")], check=True, timeout=900)
print("Setup finished. Use Runtime > Restart session, then run the next tests/Drive cell. Do not repeat upload.")


In [ ]:
from pathlib import Path
import subprocess, sys, json
root = Path("/content/judge_colab")
code_dir, bundle = root / "code", root / "bundle"
result = subprocess.run([sys.executable, str(code_dir / "test_colab.py")], capture_output=True, text=True, timeout=300)
print(result.stdout, end="")
print(result.stderr, end="")
result.check_returncode()  # A failed test still stops execution before GPU work.
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Choose a BF16-capable CUDA GPU."
print(torch.cuda.get_device_name(), "CUDA", torch.version.cuda)
from google.colab import drive
drive.mount("/content/drive")
# Change this name for a new experiment. Existing runs are never overwritten.
run_root = Path("/content/drive/MyDrive/judge_corrected_20261007")
run_root.mkdir(parents=True, exist_ok=True)

def launch(mode, output, minutes, *arguments):
    subprocess.run([sys.executable, str(code_dir / "launch.py"), "--minutes", str(minutes),
                    "--output", str(output), mode, "--bundle", str(bundle), *map(str, arguments)], check=True)


## First: check that this GPU can run the recipe

The probe downloads the pinned original Qwen weights, runs two updates, exercises a long prompt, saves Adam/random state, and verifies an exact restart. It has a **20-minute total deadline**, including download. No automatic retry. If it fails, keep the log; do not enable the sweep. A later manual attempt needs a reviewed cause and a new output folder.

GPU time is billed by Colab while the runtime is connected. Disconnect when finished; these scripts do not purchase credits or manage your account.

In [ ]:
RUN_PROBE = False  # Change to True when ready to use GPU time.
if RUN_PROBE:
    launch("probe", run_root / "probe", 20)
    print((run_root / "probe/result.json").read_text())
else:
    print("Probe prepared; not started. Set RUN_PROBE=True to begin.")


## Then: repeat the three learning rates with the corrected optimizer

Keep hard labels, JSON, all 36 layers, and no length weighting fixed. Each rate sees the same first 600 pairs, both orders, over 150 updates. The **200-pair training-side monitor** selects among updates 100 and 150; validation is not used for selection. Default limit: 120 minutes per rate, **six hours total at most**. Inspect the probe timings before setting this limit. A failure stops the sweep without retrying.

The earlier MLX winner `2e-5` is historical; it is not automatically the winner after these changes.

In [ ]:
RUN_SWEEP = False
MINUTES_PER_RATE = 120
if RUN_SWEEP:
    subprocess.run([sys.executable, str(code_dir / "sweep.py"), "--bundle", str(bundle),
                    "--probe", str(run_root / "probe"), "--output", str(run_root / "sweep"),
                    "--minutes-per-rate", str(MINUTES_PER_RATE)], check=True)
    print((run_root / "sweep/selection.json").read_text())


## Finally: compare both models on all validation questions

These cells each permit 60 minutes. Accuracy uses 311 clear labels; all 85 ties stay diagnostic. Reports include log-loss, AUC, grouped 95% intervals, the paired test, preference strength, shorter winners, answer order, and accuracy versus coverage.

The new unchanged-model result is the reference for this CUDA/FP32 recipe. Keep baseline v2's original BF16-head result locked as history. These development measurements do not establish calibrated confidence or final-test performance.

In [ ]:
RUN_FULL_VALIDATION = False
if RUN_FULL_VALIDATION:
    selected = json.loads((run_root / "sweep/selection.json").read_text())["selected"]
    launch("evaluate", run_root / "baseline_fp32", 60)
    launch("evaluate", run_root / "selected_fp32", 60, "--checkpoint", selected["checkpoint"],
           "--baseline", run_root / "baseline_fp32")
    print((run_root / "selected_fp32/full_report.json").read_text())


## What happens after this notebook?

Review the reports before launching the longer 2,000-pair run. It takes 500 updates for one pass or up to 1,000 for two, with monitor-based stopping. Then compare **hard / smoothed / vote** targets on the same clear pairs; adding tie-labelled pairs is a separate experiment. Test length balancing and readable formatting separately. The scripts support those options, but this notebook does not launch those experiments.

Checkpoints store adapters, Adam state, random state, and fingerprints. An explicit restart uses `train.py --resume` through `launch.py`, with the original arguments and latest complete checkpoint. Interrupted logs are retained; elapsed budgets do not reset. No automatic restart occurs.

When the final model is locked, use calibration data to fit temperature and choose abstention, then use the final test once. Those splits are intentionally absent here. Read `README.md` inside the package for the current limits.